# Interactive Packing Simulator Demo

This notebook launches the manual packing simulator inside Jupyter. The simulator runs as a small local HTTP server and is embedded below with an `IFrame`.

The goal is not only to run the tool, but also to explain two implementation details that are easy to miss when using it: how grid-mode candidate points are generated, and how the stability check/update follows Algorithm 1 and Algorithm 2 in the paper.


## 1. Start The Simulator

Run this cell once. It starts the simulator in a background thread and chooses a free local port automatically.

In [5]:
import socket
import sys
from http.server import ThreadingHTTPServer
from pathlib import Path
from threading import Thread

from IPython.display import IFrame, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "interactive_simulator_app").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from interactive_simulator_app.server import SimulatorRequestHandler
from interactive_simulator_app.simulator import InteractivePackingSimulator


def find_free_port(host="127.0.0.1"):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind((host, 0))
        return sock.getsockname()[1]


HOST = "127.0.0.1"
PORT = find_free_port(HOST)

SimulatorRequestHandler.simulator = InteractivePackingSimulator(
    seed=0,
    ds_name="random",
    buffer_capacity=12,
    container_size=(600, 600, 600),
    k_placement=80,
    buffer_space=0,
    remove_inscribed_ems=False,
)

simulator_server = ThreadingHTTPServer((HOST, PORT), SimulatorRequestHandler)
simulator_thread = Thread(target=simulator_server.serve_forever, daemon=True)
simulator_thread.start()

SIMULATOR_URL = f"http://{HOST}:{PORT}"
print("Simulator running at:", SIMULATOR_URL)

Simulator running at: http://127.0.0.1:45767
[simulator] 127.0.0.1 - "GET / HTTP/1.1" 200 -
[simulator] 127.0.0.1 - "GET /static/style.css HTTP/1.1" 200 -
[simulator] 127.0.0.1 - "GET /static/simulator.js HTTP/1.1" 200 -
[simulator] 127.0.0.1 - "GET /state HTTP/1.1" 200 -
[simulator] 127.0.0.1 - "POST /place HTTP/1.1" 200 -
[simulator] 127.0.0.1 - "POST /place HTTP/1.1" 200 -
[simulator] 127.0.0.1 - "POST /place HTTP/1.1" 200 -
[simulator] 127.0.0.1 - "POST /place HTTP/1.1" 200 -
[simulator] 127.0.0.1 - "POST /place HTTP/1.1" 200 -
[simulator] 127.0.0.1 - "POST /place HTTP/1.1" 200 -
[simulator] 127.0.0.1 - "GET / HTTP/1.1" 200 -
[simulator] 127.0.0.1 - "GET /static/style.css HTTP/1.1" 200 -
[simulator] 127.0.0.1 - "GET /static/simulator.js HTTP/1.1" 200 -
[simulator] 127.0.0.1 - "GET /state HTTP/1.1" 200 -
[simulator] 127.0.0.1 - "POST /place HTTP/1.1" 200 -


## 2. Grid Mode Candidate Generation

The simulator has two placement modes. `Anchors` mode shows the feasible EMS anchor points produced by the packing environment. `Grid` mode is more exploratory: it samples many possible `(x, y)` positions on a regular grid and lets the stability validator decide whether each position can support the current item.

In the current implementation, only the set of grid points that do not cause overlapping or exceeding the container are presented. 
Visually, solid teal previews are valid candidates. Transparent red previews are rejected candidates. This makes grid mode useful for inspecting the difference between geometric placeability and structural stability.


## 3. Stability Check And Feasibility Map Update

The simulator's stability logic follows the paper's structural stability validation idea in Algorithm 1 and Algorithm 2. The paper describes the method using a height map `HM` and a feasibility map `FM`. The code implements this mainly in `packing_env/heu_stable.py`.

### Algorithm 1: Structural Stability Check

For a candidate item placement, Algorithm 1 checks whether the item's center of gravity region is contained in the load-bearable support polygon. In code, this corresponds to `Heu_Stable._convex_hull_validate()`.

In the simulator's grid mode, this check is called after a candidate passes the basic container/collision test. In the normal environment action pipeline, `PackingEnv.get_stable_lps_mask()` applies the same check to EMS candidates and uses the result as the action mask.

### Algorithm 2: Structural Stability Update

After an item is actually packed, Algorithm 2 updates the feasibility map so future items only use load-bearable regions. In code, this corresponds to `Heu_Stable.update()` and `_compute_updated_feasible_map()`.

This is why the yellow/golden support patch matters in the simulator: it is not just a drawing. It is the visual counterpart of the load-bearable region that future stability checks can use.


## 4. Use The Simulator

The embedded simulator supports anchor placement, grid placement, support visualization, utilization display, buffer visualization, and editable container size.

Try this sequence:

1. Start in `Anchors` mode and hover/click the green markers.
2. Switch to `Grid` mode and hover over grid candidates.
3. Toggle `Support` to show or hide the golden load-bearable support patches.
4. Place several items and watch how the support patches and valid grid candidates change.


In [6]:
display(IFrame(src=SIMULATOR_URL, width="100%", height=1300))

## 5. Stop The Simulator

Run this cleanup cell when you are done.


In [7]:
simulator_server.shutdown()
simulator_server.server_close()
print("Simulator stopped.")

Simulator stopped.
